# View fetal brain segmentation (`dseg`)

This notebook loads `mri_gz/sub-001_rec-mial_dseg.nii.gz` (FeTA-style discrete tissue labels) and displays it as:

- orthogonal mid-slices
- an overlay on the matching T2-weighted MRI
- an interactive slice slider

**Labels (FeTA):** 0 background · 1 extra-axial CSF · 2 gray matter · 3 white matter · 4 ventricles · 5 cerebellum · 6 deep gray matter · 7 brainstem


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import nibabel as nib
import numpy as np
from matplotlib.colors import ListedColormap

DSEG_PATH = Path("mri_gz/sub-001_rec-mial_dseg.nii.gz")
T2W_PATH = Path("mri_gz/sub-001_rec-mial_T2w.nii.gz")

LABEL_NAMES = {
    0: "background",
    1: "extra-axial CSF",
    2: "gray matter",
    3: "white matter",
    4: "ventricles",
    5: "cerebellum",
    6: "deep gray matter",
    7: "brainstem",
}

LABEL_COLORS = [
    (0, 0, 0, 0),          # 0 background (transparent for overlays)
    (0.12, 0.47, 0.71, 1), # 1 CSF — blue
    (0.84, 0.15, 0.16, 1), # 2 GM — red
    (0.17, 0.63, 0.17, 1), # 3 WM — green
    (0.58, 0.40, 0.74, 1), # 4 ventricles — purple
    (1.00, 0.50, 0.05, 1), # 5 cerebellum — orange
    (0.89, 0.47, 0.76, 1), # 6 deep GM — pink
    (0.74, 0.74, 0.13, 1), # 7 brainstem — yellow
]
SEG_CMAP = ListedColormap(LABEL_COLORS)


In [ ]:
dseg_img = nib.load(DSEG_PATH)
dseg = np.asarray(dseg_img.dataobj, dtype=np.int16)

print(f"File:   {DSEG_PATH}")
print(f"Shape:  {dseg.shape}")
print(f"Voxel:  {dseg_img.header.get_zooms()}")
print(f"Labels: {np.unique(dseg)}")
print()
for lab in np.unique(dseg):
    n = int((dseg == lab).sum())
    print(f"  {int(lab):d}  {LABEL_NAMES.get(int(lab), '?'):20s}  {n:,} voxels")


## Orthogonal mid-slices

The volume is shown in sagittal, coronal, and axial views at the centre of the brain.


In [ ]:
def mid_slices(volume):
    x, y, z = (s // 2 for s in volume.shape)
    return {
        "Sagittal (x)": np.rot90(volume[x, :, :]),
        "Coronal (y)": np.rot90(volume[:, y, :]),
        "Axial (z)": np.rot90(volume[:, :, z]),
    }


def label_legend():
    return [
        mpatches.Patch(color=LABEL_COLORS[i], label=f"{i}: {name}")
        for i, name in LABEL_NAMES.items()
        if i > 0
    ]


fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, (title, sl) in zip(axes, mid_slices(dseg).items()):
    ax.imshow(sl, cmap=SEG_CMAP, vmin=0, vmax=7, interpolation="nearest")
    ax.set_title(title)
    ax.axis("off")

fig.legend(handles=label_legend(), loc="lower center", ncol=4, frameon=False)
fig.suptitle("sub-001 discrete segmentation (dseg)", fontsize=14)
fig.tight_layout(rect=[0, 0.08, 1, 0.95])
plt.show()


## Overlay on T2-weighted MRI

The matching anatomical scan is `sub-001_rec-mial_T2w.nii.gz`. Labels are drawn on top so you can check how they sit on the brain tissue.


In [ ]:
t2w_img = nib.load(T2W_PATH)
t2w = np.asarray(t2w_img.dataobj, dtype=np.float32)
print(f"T2w shape: {t2w.shape}  (should match dseg {dseg.shape})")

overlay_cmap = ListedColormap(LABEL_COLORS)
overlay = np.ma.masked_where(dseg == 0, dseg)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, key in zip(axes, mid_slices(t2w)):
    anat = mid_slices(t2w)[key]
    seg = mid_slices(overlay)[key]
    ax.imshow(anat, cmap="gray", interpolation="nearest")
    ax.imshow(seg, cmap=overlay_cmap, vmin=0, vmax=7, interpolation="nearest", alpha=0.55)
    ax.set_title(key)
    ax.axis("off")

fig.legend(handles=label_legend(), loc="lower center", ncol=4, frameon=False)
fig.suptitle("T2w with dseg overlay", fontsize=14)
fig.tight_layout(rect=[0, 0.08, 1, 0.95])
plt.show()


## Interactive slice viewer

Use the sliders to scroll through the volume. Enable **overlay** to draw labels on the T2w scan.


In [ ]:
from ipywidgets import interact, IntSlider, Dropdown, Checkbox


def slice_view(plane: str, index: int, overlay_on_t2w: bool):
    if plane == "sagittal":
        anat, labels = t2w[index, :, :], dseg[index, :, :]
    elif plane == "coronal":
        anat, labels = t2w[:, index, :], dseg[:, index, :]
    else:
        anat, labels = t2w[:, :, index], dseg[:, :, index]

    anat = np.rot90(anat)
    labels = np.rot90(labels)

    fig, ax = plt.subplots(figsize=(6, 6))
    if overlay_on_t2w:
        ax.imshow(anat, cmap="gray", interpolation="nearest")
        masked = np.ma.masked_where(labels == 0, labels)
        ax.imshow(masked, cmap=overlay_cmap, vmin=0, vmax=7, interpolation="nearest", alpha=0.55)
        ax.set_title(f"T2w + dseg  |  {plane} slice {index}")
    else:
        ax.imshow(labels, cmap=SEG_CMAP, vmin=0, vmax=7, interpolation="nearest")
        ax.set_title(f"dseg  |  {plane} slice {index}")
    ax.axis("off")
    plt.show()


interact(
    slice_view,
    plane=Dropdown(options=["axial", "coronal", "sagittal"], value="axial"),
    index=IntSlider(value=dseg.shape[2] // 2, min=0, max=dseg.shape[2] - 1, step=1),
    overlay_on_t2w=Checkbox(value=True, description="overlay on T2w"),
);
